In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import wikipediaapi


wiki = wikipediaapi.Wikipedia('PEDSearch/1.0', "en")

SEED_CATEGORIES = [
    "Category:Ergogenic_aids",                    # broad: all PEDs legal + illegal
    "Category:Anabolic_steroids",                 # includes therapeutic-use compounds
    "Category:Stimulants",                        # caffeine → amphetamines spectrum
    "Category:Peptide_hormones",                  # HGH, EPO, IGF-1
    "Category:Beta-2_agonists",                   # asthma drugs, some banned in sport
    "Category:Dietary_supplements",               # creatine, BCAAs — all legal
    "Category:Blood_doping_in_sport",             # transfusions, EPO use context
]

In [3]:
cat = wiki.page(SEED_CATEGORIES[0])

members = cat.categorymembers

members['Adderall']

Adderall (lang: en, variant: None, id: ??, ns: 0)

In [4]:
adderal_page = wiki.page(members['Adderall'].title)

In [5]:
categories = cat.categories
len(categories), len(cat.categorymembers)

(4, 21)

In [6]:
for section in adderal_page.sections:
    print(section.full_text)

<bound method WikipediaPageSection.full_text of Section: Uses (1):

Subsections (3):
Section: Medical (2):
Adderall is indicated for the treatment of attention deficit hyperactivity disorder (ADHD) and narcolepsy.
Subsections (3):
Section: Attention Deficit Hyperactivity Disorder (3):
Long-term amphetamine exposure at sufficiently high doses in some animal species is known to produce abnormal dopamine system development or nerve damage, but, in humans with ADHD, long-term use of pharmaceutical amphetamines at therapeutic doses appears to improve brain development and nerve growth. Reviews of magnetic resonance imaging (MRI) studies suggest that long-term treatment with amphetamine decreases abnormalities in brain structure and function found in subjects with ADHD, and improves function in several parts of the brain, such as the right caudate nucleus of the basal ganglia.
Reviews of clinical stimulant research have established the safety and effectiveness of long-term continuous ampheta

In [7]:
from day_three.dataset.dataset import DrugRawSubtranceDataset

dataset = DrugRawSubtranceDataset(SEED_CATEGORIES)

/Users/andrei/projects/qdrant-essesials/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
substances = []
for _ in range(1000):
    new_element = next(dataset)
    print("New element is: ", new_element)
    substances.append(new_element)

New element is:  name='Performance-enhancing substance' description='Performance-enhancing substances (PESs), also known as performance-enhancing drugs (PEDs), are substances that are used to improve any form of activity performance in humans.\nMany substances, such as anabolic steroids, can be used to improve athletic performance and build muscle, which in most cases is considered cheating by organized athletic organizations. This usage is often referred to as doping. Athletic performance-enhancing substances are sometimes referred to as ergogenic aids. Cognitive performance-enhancing drugs, commonly called nootropics, are sometimes used by students to improve academic performance. Performance-enhancing substances are also used by military personnel to enhance combat performance.' drug_category='Category:Ergogenic_aids' wada_status='banned' wada_category=<WadaCategory.S1: ('S1', 'Anabolic Agents', <ProhibitionTiming.ALL_TIMES: 'all_times'>)> side_effects=None
New element is:  name='Ad

StopIteration: 

In [10]:
substances[0].name, substances[0].drug_category, substances[0].wada_category, substances[0].wada_status, substances[0].side_effects

('Performance-enhancing substance',
 'Category:Ergogenic_aids',
 <WadaCategory.S1: ('S1', 'Anabolic Agents', <ProhibitionTiming.ALL_TIMES: 'all_times'>)>,
 'banned',
 None)

In [11]:
len(substances)

791

In [12]:
import os

from qdrant_client import QdrantClient

url = os.getenv('QDRANT_URL')
api_key = os.getenv('QDRANT_API_KEY')


client = QdrantClient(url=url, api_key=api_key)


/var/folders/pq/gsx7ryn905s1bypf32zs4k980000gn/T/ipykernel_55955/3378853019.py:9: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(url=url, api_key=api_key)


In [13]:
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='midjourney')])

In [31]:
from day_three.vector_db.qdrant_wrapper import HybridSearchCollection

drug_search = HybridSearchCollection('sport-drug')

drug_search.collection

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10115.99it/s]
/Users/andrei/projects/qdrant-essesials/src/day_three/vector_db/qdrant_wrapper.py:17: UserWarning: Api key is used with an insecure connection.
  Prefetch,


<bound method HybridSearchCollection.collection of <day_three.vector_db.qdrant_wrapper.HybridSearchCollection object at 0x1341e2fd0>>

In [32]:
drug_search.create_collection()

True

In [33]:
drug_search.add_data_points(substrances=substances)

In [34]:
drug_search.hybrid_search('testosterone for olympics')

[ScoredPoint(id='a1c74f1f-f61d-46b1-8b30-0e616e19d3e1', version=8, score=0.5, payload={'name': 'Gonadotropin-releasing hormone', 'description': 'Gonadotropin-releasing hormone (GnRH) is a releasing hormone responsible for the release of follicle-stimulating hormone (FSH) and luteinizing hormone (LH) from the anterior pituitary. GnRH is a tropic peptide hormone synthesized and released from GnRH neurons within the hypothalamus. GnRH is inhibited by testosterone. The peptide belongs to gonadotropin-releasing hormone family. It constitutes the initial step in the activation of hypothalamic–pituitary–gonadal axis.', 'drug_category': 'Category:Peptide_hormones', 'wada_status': 'banned', 'wada_category': 'S2 - Peptide Hormones, Growth Factors, Related Substances and Mimetics', 'side_effects': None}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='2e2a50f7-1762-421c-9c61-dc0df9b761ae', version=1, score=0.5, payload={'name': 'Performance-enhancing substance', 'description': 'P

In [35]:
drug_search.hybrid_search("caffeine energy drink")[0].payload

{'name': 'Caffeine',
 'description': 'Caffeine is a central nervous system (CNS) stimulant of the methylxanthine class and is the most commonly consumed psychoactive substance globally. It is mainly used for its eugeroic (wakefulness promoting), ergogenic (physical performance-enhancing), or nootropic (cognitive-enhancing) properties; it is also used recreationally or in social settings. Caffeine acts by blocking the binding of adenosine at a number of adenosine receptor types, inhibiting the centrally depressant effects of adenosine and enhancing the release of acetylcholine. Caffeine has a three-dimensional structure similar to that of adenosine, which allows it to bind and block its receptors. Caffeine also increases cyclic AMP levels through nonselective inhibition of phosphodiesterase, increases calcium release from intracellular stores, and antagonizes GABA receptors, although these mechanisms typically occur at concentrations beyond usual human consumption.\nCaffeine is a bitter

In [36]:
drug_search.hybrid_search("morphine painkiller opioid narcotic")[0].payload

{'name': 'Amiphenazole',
 'description': 'Amiphenazole (Daptazile) is a respiratory stimulant traditionally used as an antidote for barbiturate or opiate overdose, usually in combination with bemegride, as well as poisoning from other sedative drugs and treatment of respiratory failure from other causes. It was considered particularly useful as it could counteract the sedation and respiratory depression produced by morphine but with less effect on analgesia. It is still rarely used in medicine in some countries, although it has largely been replaced by more effective respiratory stimulants such as doxapram and specific opioid antagonists such as naloxone.\n\n\n== References ==',
 'drug_category': 'Category:Stimulants',
 'wada_status': 'conditional',
 'wada_category': 'S7 - Narcotics',
 'side_effects': None}

In [37]:
drug_search.sparse_search("urine substitution catheterization sample")[0].payload

{'name': 'Pentylone',
 'description': 'Pentylone, also known as β-ketoN-methyl-1,3-benzodioxolylpentanamine (bk-MBDP), is a stimulant developed in the 1960s. It is a substituted cathinone that has been identified in some samples of powders sold as "NRG-1", along with varying blends of other cathinone derivatives including flephedrone, MDPBP, MDPV, and 4-MePPP. \nIt was also found in combination with 4-MePPP being sold as "NRG-3". Reports indicate side effects include paranoia, agitation, and insomnia, with effects lasting for several days at high doses.',
 'drug_category': 'Category:Stimulants',
 'wada_status': 'banned',
 'wada_category': 'M2 - Chemical and Physical Manipulation',
 'side_effects': None}

In [38]:
results = drug_search.hybrid_search("morphine opioid", query_limit=20)
names = [r.payload["name"] for r in results]
print(names)

['Amiphenazole', 'Speedball (drug)', 'Tebanicline', 'Fenmetramide', 'Epiboxidine', 'ODMA (drug)', 'Fenozolone', '(+)-CPCA', 'D-IX', 'Indatraline', 'Feprosidnine', 'Benocyclidine', 'PNU-99,194', '2,3,4,5-Tetrahydro-1,5-methano-1H-3-benzazepine', 'Fenisorex', 'Club drug', 'Fenproporex', '3-Methylmethcathinone', 'SeDMA', 'Dextroamphetamine']


In [39]:
athlete = [
    "something to recover faster after hard training",
    "supplement to boost energy before competition without getting banned"
]

coach = [
    "beta-2 agonist bronchodilator therapeutic use exemption competition",
    "anabolic androgenic steroid prohibited at all times S1 detection window"
]

In [40]:
results = {
    'hybrid': drug_search.hybrid_search(athlete[0])[0],
    'dense': drug_search.dense_search(athlete[0])[0],
    'sparse': drug_search.sparse_search(athlete[0])[0],
}

results

{'hybrid': ScoredPoint(id='ce2d0361-4d56-4603-90f8-f19d6ff8362f', version=1, score=0.6666667, payload={'name': 'Β-Hydroxy β-methylbutyric acid', 'description': 'β-Hydroxy β-methylbutyric acid (HMB), otherwise known as its conjugate base, β-hydroxy β-methylbutyrate, is a naturally produced substance in humans that is used as a dietary supplement and as an ingredient in certain medical foods that are intended to promote wound healing and provide nutritional support for people with muscle wasting due to cancer or HIV/AIDS. In healthy adults, supplementation with HMB has been shown to increase exercise-induced gains in muscle size, muscle strength, and lean body mass, reduce skeletal muscle damage from exercise, improve aerobic exercise performance, and expedite recovery from exercise; in trained and competitive athletes, evidence is mixed on whether it meaningfully augments resistance training–induced gains in lean mass and strength. Medical reviews and meta-analyses indicate that HMB sup

In [41]:
[(results[result_key].payload['name'], result_key) for result_key in results if results[result_key].payload is not None]

[('Β-Hydroxy β-methylbutyric acid', 'hybrid'),
 ('Β-Hydroxy β-methylbutyric acid', 'dense'),
 ('Eggshell membrane', 'sparse')]

In [42]:
results = {
    'hybrid': drug_search.hybrid_search(athlete[1])[0],
    'dense': drug_search.dense_search(athlete[1])[0],
    'sparse': drug_search.sparse_search(athlete[1])[0],
}

results

{'hybrid': ScoredPoint(id='8981c552-2c3f-404e-b9a4-78fdc00561ee', version=12, score=0.75, payload={'name': 'Sports drink', 'description': 'Sports drinks, also known as electrolyte drinks, are non-caffeinated functional beverages whose stated purpose is to help athletes replace water, electrolytes, and energy before, during and (especially) after training or competition.\nSports drinks are classified primarily by their osmolarity into hypotonic, isotonic, and hypertonic types. According to an investigative report in 2012, the evidence was limited pertaining to the efficacy of use of commercial sports drinks for sports and fitness performance. Consuming these drinks in excess or when unnecessary may negatively affect health or performance, and some ingredients, such as sugar, may not be suitable for certain conditions. A 2021 systematic review found that hypotonic carbohydrate-electrolyte solutions, formulated with lower sugar and low-to-moderate sodium, are generally most effective at m

In [43]:
[(results[result_key].payload['name'], result_key) for result_key in results if results[result_key].payload is not None]

[('Sports drink', 'hybrid'),
 ('Sports drink', 'dense'),
 ('Methylhexanamine', 'sparse')]

In [44]:
results = {
    'hybrid': drug_search.hybrid_search(coach[0])[0],
    'dense': drug_search.dense_search(coach[0])[0],
    'sparse': drug_search.sparse_search(coach[0])[0],
}

results

{'hybrid': ScoredPoint(id='25fa272f-9a94-4e7f-9e4a-f5af41d13dba', version=1, score=1.0, payload={'name': 'Theophylline/ephedrine', 'description': 'Theophylline ephedrine (INNTooltip International Nonproprietary Name), or theophylline/ephedrine, sold under the brand name Franol among others, is a fixed-dose combination formulation of theophylline, an adenosine receptor antagonist, and ephedrine, a norepinephrine releasing agent and indirectly acting sympathomimetic agent, which has been used as a bronchodilator in the treatment of asthma and as a nasal decongestant. It was first studied and used to treat asthma in the 1930s or 1940s and combinations of the two drugs subsequently became widely used. A ratio of 5:1 theophylline to ephedrine is usually used in combinations of the drugs. Later research found that the combination was no more effective for asthma than theophylline alone but produced more side effects.\nCombinations of theophylline, ephedrine, and phenobarbital (brand name Ted

In [45]:
[(results[result_key].payload['name'], result_key) for result_key in results if results[result_key].payload is not None]

[('Theophylline/ephedrine', 'hybrid'),
 ('Theophylline/ephedrine', 'dense'),
 ('Theophylline/ephedrine', 'sparse')]

In [46]:
results = {
    'hybrid': drug_search.hybrid_search(coach[1])[0],
    'dense': drug_search.dense_search(coach[1])[0],
    'sparse': drug_search.sparse_search(coach[1])[0],
}

results

{'hybrid': ScoredPoint(id='2e2a50f7-1762-421c-9c61-dc0df9b761ae', version=1, score=0.8333334, payload={'name': 'Performance-enhancing substance', 'description': 'Performance-enhancing substances (PESs), also known as performance-enhancing drugs (PEDs), are substances that are used to improve any form of activity performance in humans.\nMany substances, such as anabolic steroids, can be used to improve athletic performance and build muscle, which in most cases is considered cheating by organized athletic organizations. This usage is often referred to as doping. Athletic performance-enhancing substances are sometimes referred to as ergogenic aids. Cognitive performance-enhancing drugs, commonly called nootropics, are sometimes used by students to improve academic performance. Performance-enhancing substances are also used by military personnel to enhance combat performance.', 'drug_category': 'Category:Ergogenic_aids', 'wada_status': 'banned', 'wada_category': 'S1 - Anabolic Agents', 'si

In [47]:
[(results[result_key].payload['name'], result_key) for result_key in results if results[result_key].payload is not None]

[('Performance-enhancing substance', 'hybrid'),
 ('Performance-enhancing substance', 'dense'),
 ('Prasterone', 'sparse')]

---
[Day 3] Building a Hybrid Search Engine

High-Level Summary

- Domain: Hybrid search over WADA-prohibited substances — given a natural-language query (athlete or coach), surface relevant drugs with their ban status and
side effects.
- Winner: Hybrid (RRF) worked best because athlete queries use goal-driven language with no domain vocabulary ("recover faster after training"), which dense
handles well, while coach queries use exact WADA codes and chemi well. Neither alone covers both personas.

---
Reproducibility

- Collection: wada-substances
- Models: dense=[all-MiniLM-L6-v2, 384 dim], sparse=[Qdrant/bm25
- Dataset: ~150–200 items from Category:Stimulants + Category:Ergogenic_aids (snapshot: 2026-06-08)
▎ Note: opioids, anabolic steroids, cannabinoids, glucocorticoid testosterone absent.

---
Settings (today)

- Fusion: RRF, k=60 (Qdrant default), prefetch_limit=20 per leg
- Search: HNSW ef — default (not explicitly set)
- Sparse encoding: BM25 (Qdrant/bm25), IDF modifier enabled, tokenisation and stop-word removal handled by fastembed internally

---
Head-to-Head (demo query: "something to recover faster after har

- Dense top-3: 1) HMB, 2) (not recorded), 3) (not recorded)
- Sparse top-3: 1) Eggshell membrane, 2) (not recorded), 3) (not recorded)
- Hybrid top-3: 1) HMB, 2) (not recorded), 3) (not recorded)

▎ Run dense_search, sparse_search, hybrid_search with query_limi

---
Latency                                                                                                                                                   
- Dense: not measured                                                                                                                                     - Sparse: not measured
- Hybrid (RRF): not measured                                                                                                                              
▎ Suggested snippet:                                                                                                                                      ▎ import time, statistics
▎ times = []                                                                                                                                              ▎ for _ in range(50):
▎     t = time.perf_counter()                                                                                                                             ▎     drug_search.hybrid_search("recover faster after training")
▎     times.append((time.perf_counter() - t) * 1000)                                                                                                      ▎ print(f"avg={statistics.mean(times):.1f}ms  P95={statistics.qums")
                                                                                                                                                          ---
Why these won                                                                                                                                             
- Dense captured athlete intent via semantic similarity (no exact keywords needed); BM25 locked onto WADA codes and chemical names in coach queries — neitalone handles both, RRF fusion combines their ranked lists withon.
                                                                                                                                                          ---
Surprise                                                                                                                                                  
- SimpleSparsePointGenerator (raw TF, no stop-word filtering) was surfacing cocaine as the top result for recovery queries — its Wikipedia article is 10× longer than most entries, so common tokens like "after", "for", eptively high sparse score. Replacing with Qdrant/bm25 fixed itimmediately.

---
Next step

- Add a ProductSuggester post-search layer that maps top substanducts via SerpApi Google Shopping — e.g. Prasterone → DHEAsupplements, Sport energy drink → Gatorade.